# D0 — document identity, original bytes and three clocks

This is the **first checkpoint** of #11, not completion of D0 or a document ingestion pipeline.
It uses only core contracts: no Git, graph, LangChain, model credentials or network.

We will distinguish a logical document, a captured revision, reusable bytes, collection
membership and a chunk occurrence. Then we will show why old source validity does not
establish old knowledge. See `docs/DOCUMENT_CONTRACTS.md` for the specification.

In [ ]:
from datetime import UTC, datetime

from pydantic import ValidationError

from synirmos.core.contracts import ArtifactSpec, SourceKey, json_fingerprint
from synirmos.core.documents import (
    ChunkContent,
    ChunkOccurrence,
    ContentReference,
    DocumentBatch,
    DocumentCollection,
    DocumentLimits,
    DocumentRevision,
    SourceSpan,
    TextDocument,
)
from synirmos.core.temporal import ReadPoint, RevisionTiming, ValidInterval

VALID_FROM = datetime(2026, 11, 1, tzinfo=UTC)
CAPTURED_AT = datetime(2026, 11, 28, tzinfo=UTC)
prefix = b"# Notes\r\n\r\n"
line = "Ada wrote café.\r\n".encode()
raw = prefix + line + line
content = ContentReference.from_bytes(raw)
timing = RevisionTiming(
    valid=ValidInterval(start=VALID_FROM),
    observed_at=CAPTURED_AT,
    info_horizon=CAPTURED_AT,
)
documents = [
    TextDocument(key=SourceKey(source_instance=source, native_key="notes"), title="Notes")
    for source in ("memory:alice", "memory:bob")
]
revisions = [DocumentRevision(document=doc, content=content, timing=timing) for doc in documents]
assert documents[0].document_id != documents[1].document_id
assert revisions[0].revision_id != revisions[1].revision_id
assert revisions[0].content == revisions[1].content
print("Same bytes; distinct source documents and captured revisions.")

## Membership does not own identity; replay does not create a new observation

One document can belong to two collections. Neither membership nor an incidental
processing run changes its identity. A retry restores the original capture timestamps;
calling a new clock would instead describe another observation. Title changes do not
change logical identity, but captured metadata does participate in revision identity.

In [ ]:
memberships = [
    DocumentCollection(
        key=SourceKey(source_instance="collections:demo", native_key=name),
        document_ids=(documents[0].document_id,),
    )
    for name in ("team", "research")
]
assert memberships[0].collection_id != memberships[1].collection_id
assert memberships[0].document_ids == memberships[1].document_ids
retry = DocumentRevision.model_validate_json(revisions[0].model_dump_json())
assert retry == revisions[0]
assert retry.revision_id == revisions[0].revision_id
renamed = TextDocument(key=documents[0].key, title="Renamed notes")
assert renamed.document_id == documents[0].document_id
print("Two memberships; one logical document; retry retains captured identity.")

## Source validity, observation and justified knowledge are independent

Our source says the text is valid from November 1, but we captured it on November 28.
That does not prove the content was available to this application on November 1.
Backdating knowledge requires availability evidence that a source adapter must verify.
Merely supplying an evidence ID is **not authentication or permission**. None of these
times is a database commit timestamp. `ReadPoint.includes` checks one revision's time
eligibility only; it does not choose a version, resolve supersession or enforce access.

In [ ]:
yesterday_view = ReadPoint(valid_time=VALID_FROM, info_horizon=VALID_FROM)
captured_view = ReadPoint(valid_time=VALID_FROM, info_horizon=CAPTURED_AT)
assert not yesterday_view.includes(timing)
assert captured_view.includes(timing)
try:
    RevisionTiming(valid=timing.valid, observed_at=CAPTURED_AT, info_horizon=VALID_FROM)
except ValidationError:
    print("Rejected unsupported backdating, despite the old validity start.")
else:
    raise AssertionError("missing availability evidence must reject")
assert not ValidInterval(start=VALID_FROM, end=CAPTURED_AT).contains(CAPTURED_AT)

## Original-byte spans, not decoded character positions

This fixture contains UTF-8, CRLF and a repeated line. We explicitly choose the second
line using the known byte structure, not an ambiguous search for its text. A real parser
must supply source maps when headings, whitespace or escapes are transformed. Span
extraction verifies the complete original content reference before slicing.

In [ ]:
start = len(prefix) + len(line)
span = SourceSpan(
    revision_id=revisions[0].revision_id, content=content, start=start, end=start + len(line)
)
assert span.extract(raw) == line
assert len(line) > len(line.decode("utf-8"))
print(f"Second line: bytes [{span.start}, {span.end}), {span.extract(raw)!r}")

## Reuse chunk content without merging its occurrences or access

A chunker specification identifies the algorithm/library version and exact config.
Identical chunk bytes under that specification may be reused across revisions, but
each original placement has its own occurrence identity. These values do not perform
chunking or grant permission to retrieve another source's content.

In [ ]:
spec = ArtifactSpec(
    implementation="example:line-splitter",
    implementation_version="1",
    config_fingerprint=json_fingerprint({"preserve_crlf": True, "overlap": 0}),
)
chunk = ChunkContent(content=ContentReference.from_bytes(line), chunker=spec)
occurrences = [
    ChunkOccurrence(
        chunk=chunk,
        span=SourceSpan(
            revision_id=rev.revision_id, content=content, start=start, end=start + len(line)
        ),
    )
    for rev in revisions
]
assert occurrences[0].chunk.chunk_id == occurrences[1].chunk.chunk_id
assert occurrences[0].occurrence_id != occurrences[1].occurrence_id
print("Reusable chunk content; distinct source occurrences.")

## Bounded pages are not unbounded-work guarantees

Source pages validate both revision count and conservatively summed content bytes.
Adapters must still enforce the caller's actual policy; self-declared returned limits
cannot expand it. A continuation/gap is distinct from a successful empty result.
These limits do not certify parser memory, source I/O, token usage or latency.

In [ ]:
limits = DocumentLimits(max_revisions=1, max_content_bytes=len(raw))
page = DocumentBatch(revisions=(revisions[0],), limits=limits, continuation="next:opaque")
assert DocumentBatch.model_validate_json(page.model_dump_json()) == page
assert DocumentBatch(continuation="next:opaque").continuation is not None
try:
    DocumentBatch(revisions=tuple(revisions), limits=limits)
except ValidationError:
    print("Rejected a two-revision page with a one-revision limit.")
else:
    raise AssertionError("source pages must enforce declared limits")

## What remains in D0

Graph/profile/scope/write contracts, trusted authorization intersection, context and
exposure results, read-session lifetimes, extraction candidates/assertions and retry
identity, and explicit physical-work/deadline capabilities are still #11 acceptance.
Actual text/file capture and parsing/chunking adapters belong to D2/#13.

The full gate runs this notebook and notebook 00 as ordinary code cells in fresh
processes, in order, without Jupyter. Passing this demonstration does not complete
the broader slice or prove extraction quality, authorization or scale readiness.